# Ionospheric Delay Analysis — Dataset 2
## Day 33 (2 May 2024, G3 Storm) and Day 37 (6 May 2024, G2 Storm)
LSTM and Transformer predictions, ionospheric delay plots, and full metrics.
> No diurnal analysis included.

### Imports

In [ ]:
import torch
import numpy as np
import pandas as pd
from src.preprocessing.sort_tec_data import build_sorted_dataset
from src.preprocessing.dataset import prepare_dataset
from src.configs.config import (
    BASE_DIR_2, OUTPUT_DIR_2,
    F_L1, F_L5, EVENT_THRESHOLD_PERCENTILE, DELAY_EVENT_THRESHOLDS_M,
    PLOTS_DIR_2, GENERATED_PLOTS_DIR, get_date_label,
)
from src.training.lstm_training import train_lstm
from src.training.transformer_training import train_transformer

from src.plots.loss_plots import (
    plot_lstm_loss, plot_transformer_loss,
)
from src.plots.prediction_plots import (
    plot_lstm_prediction, plot_transformer_prediction
)
from src.plots.delay_plots import (
    plot_lstm_delay_l1_l5_combined,
    plot_transformer_delay_l1_l5_combined,
)
from src.utils.iono_delay import delay_metrics, tec_to_iono_delay, compute_all_delays
from src.utils.save_results import save_delay_csv
from src.utils.metrics import (
    rmse,
    pearson_correlation,
    probability_of_detection,
    critical_success_index,
    f1_score,
    false_alarm_ratio,
)


---
# Dataset 2 — Day 33/37 Forecasting Workflow
---

### Base Directory and Output Setup — Dataset 2

In [ ]:
print(f"Base directory 2  : {BASE_DIR_2}")
print(f"Output directory 2: {OUTPUT_DIR_2}")


### Creating Sorted TEC Dataset — Dataset 2

In [ ]:
written_files_2 = build_sorted_dataset(2)
print("TEC data sorting completed — Dataset 2.")
print(f"Number of output CSV files written: {written_files_2}")
print(f"Saved sorted CSV files to: {OUTPUT_DIR_2}")


### Data Loading and Preprocessing — Dataset 2

In [ ]:
X_train_2, y_train_2, X_val_2, y_val_2, stats_2, daily_matrix_2, daily_files_2 = prepare_dataset(2)

print(f"X_train : {X_train_2.shape}   y_train : {y_train_2.shape}")
print(f"X_val   : {X_val_2.shape}     y_val   : {y_val_2.shape}")
print(f"Norm stats : {stats_2}")


```
Day mapping for Dataset 2 (41 days total):
  Day 41 = 10 May 2024  (already in newCode.ipynb)
  Day 40 =  9 May 2024  (already in newCode.ipynb)
  Day 37 =  6 May 2024  ← G2 Storm target  (this notebook)
  Day 33 =  2 May 2024  ← G3 Storm target  (this notebook)

Forecast pairs:
  Input Day 36  → Output Day 37  (6 May 2024, G2)
  Input Day 32  → Output Day 33  (2 May 2024, G3)
```

---
## LSTM Training — Dataset 2
---

In [ ]:
lstm_model_2, lstm_history_2 = train_lstm(X_train_2, y_train_2, X_val_2, y_val_2)


### LSTM Loss Curve — Dataset 2

In [ ]:
plot_lstm_loss(lstm_history_2, dataset_label="dataset2_storms", output_dir=str(PLOTS_DIR_2))


---
## Transformer Training — Dataset 2
---

In [ ]:
transformer_model_2, transformer_history_2 = train_transformer(X_train_2, y_train_2, X_val_2, y_val_2)


### Transformer Loss Curve — Dataset 2

In [ ]:
plot_transformer_loss(transformer_history_2, dataset_label="dataset2_storms", output_dir=str(PLOTS_DIR_2))


---
## Normalisation Stats
---

In [ ]:
x_mean_2, x_std_2 = stats_2["x_mean"], stats_2["x_std"]
y_mean_2, y_std_2 = stats_2["y_mean"], stats_2["y_std"]


---
# ── Day 37 — 6 May 2024 (G2 Storm) ──
---

### LSTM Forecast — Day 37 (6 May 2024, Dataset 2)

In [ ]:
# Input: Day 36 → Target: Day 37  (6 May 2024)
day36_raw_2  = daily_matrix_2[35]   # index 35 = day 36
actual37_2   = daily_matrix_2[36]   # index 36 = day 37

day36_in_2 = ((day36_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]  # (1,1440,1)

device = next(lstm_model_2.parameters()).device
src36  = torch.tensor(day36_in_2, dtype=torch.float32).to(device)

lstm_model_2.eval()
with torch.no_grad():
    pred37_norm_2 = lstm_model_2(src36).squeeze().cpu().numpy()

lstm_pred37_2 = pred37_norm_2 * y_std_2 + y_mean_2

label_37_2 = f"Actual Day 37 ({daily_files_2[36].stem.replace('_sorted', '')})"

lstm_rmse_37_2, lstm_mae_37_2 = plot_lstm_prediction(
    actual37_2, lstm_pred37_2,
    actual_label=label_37_2, target_day=37, dataset_label="dataset2",
    date_label="06_May_2024", output_dir=str(PLOTS_DIR_2),
)
print(f"LSTM Day 37 — RMSE: {lstm_rmse_37_2:.4f} TECU | MAE: {lstm_mae_37_2:.4f} TECU")


### Transformer Forecast — Day 37 (6 May 2024, Dataset 2)

In [ ]:
day36_in_t_2 = ((day36_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]

device   = next(transformer_model_2.parameters()).device
src36_t  = torch.tensor(day36_in_t_2, dtype=torch.float32).to(device)

transformer_model_2.eval()
with torch.no_grad():
    trans_pred37_norm_2 = transformer_model_2(src36_t).squeeze().cpu().numpy()

trans_pred37_2 = trans_pred37_norm_2 * y_std_2 + y_mean_2

trans_rmse_37_2, trans_mae_37_2 = plot_transformer_prediction(
    actual37_2, trans_pred37_2,
    actual_label=label_37_2, target_day=37, dataset_label="dataset2",
    date_label="06_May_2024", output_dir=str(PLOTS_DIR_2),
)
print(f"Transformer Day 37 — RMSE: {trans_rmse_37_2:.4f} TECU | MAE: {trans_mae_37_2:.4f} TECU")


### Ionospheric Delay — LSTM Forecast, Day 37 (G2 Storm)

In [ ]:
plot_lstm_delay_l1_l5_combined(
    actual37_2, lstm_pred37_2,
    target_day=37, dataset_label="dataset2",
    date_label="06_May_2024", output_dir=str(PLOTS_DIR_2)
)


### Ionospheric Delay — Transformer Forecast, Day 37 (G2 Storm)

In [ ]:
plot_transformer_delay_l1_l5_combined(
    actual37_2, trans_pred37_2,
    target_day=37, dataset_label="dataset2",
    date_label="06_May_2024", output_dir=str(PLOTS_DIR_2)
)


### Delay Error Metrics — Day 37 (6 May 2024, G2 Storm)

In [ ]:
iono_actual37_2 = tec_to_iono_delay(actual37_2)
iono_lstm37_2   = tec_to_iono_delay(lstm_pred37_2)
iono_trans37_2  = tec_to_iono_delay(trans_pred37_2)

print("Ionospheric Delay Error (L1, vertical) — Day 37 — Dataset 2 (G2 Storm)")
print("=" * 62)
delay_metrics("LSTM",        iono_lstm37_2,  iono_actual37_2)
delay_metrics("Transformer", iono_trans37_2, iono_actual37_2)


### Save Forecast CSV — Day 37 (6 May 2024)

In [ ]:
save_delay_csv(actual37_2, lstm_pred37_2, trans_pred37_2,
               output_dir=OUTPUT_DIR_2, filename="day37_06May2024_G2_ionospheric_delay.csv")


---
# ── Day 33 — 2 May 2024 (G3 Storm) ──
---

### LSTM Forecast — Day 33 (2 May 2024, Dataset 2)

In [ ]:
# Input: Day 32 → Target: Day 33  (2 May 2024)
day32_raw_2  = daily_matrix_2[31]   # index 31 = day 32
actual33_2   = daily_matrix_2[32]   # index 32 = day 33

day32_in_2 = ((day32_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]

device = next(lstm_model_2.parameters()).device
src32  = torch.tensor(day32_in_2, dtype=torch.float32).to(device)

lstm_model_2.eval()
with torch.no_grad():
    pred33_norm_2 = lstm_model_2(src32).squeeze().cpu().numpy()

lstm_pred33_2 = pred33_norm_2 * y_std_2 + y_mean_2

label_33_2 = f"Actual Day 33 ({daily_files_2[32].stem.replace('_sorted', '')})"

lstm_rmse_33_2, lstm_mae_33_2 = plot_lstm_prediction(
    actual33_2, lstm_pred33_2,
    actual_label=label_33_2, target_day=33, dataset_label="dataset2",
    date_label="02_May_2024", output_dir=str(PLOTS_DIR_2),
)
print(f"LSTM Day 33 — RMSE: {lstm_rmse_33_2:.4f} TECU | MAE: {lstm_mae_33_2:.4f} TECU")


### Transformer Forecast — Day 33 (2 May 2024, Dataset 2)

In [ ]:
day32_in_t_2 = ((day32_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]

device   = next(transformer_model_2.parameters()).device
src32_t  = torch.tensor(day32_in_t_2, dtype=torch.float32).to(device)

transformer_model_2.eval()
with torch.no_grad():
    trans_pred33_norm_2 = transformer_model_2(src32_t).squeeze().cpu().numpy()

trans_pred33_2 = trans_pred33_norm_2 * y_std_2 + y_mean_2

trans_rmse_33_2, trans_mae_33_2 = plot_transformer_prediction(
    actual33_2, trans_pred33_2,
    actual_label=label_33_2, target_day=33, dataset_label="dataset2",
    date_label="02_May_2024", output_dir=str(PLOTS_DIR_2),
)
print(f"Transformer Day 33 — RMSE: {trans_rmse_33_2:.4f} TECU | MAE: {trans_mae_33_2:.4f} TECU")


### Ionospheric Delay — LSTM Forecast, Day 33 (G3 Storm)

In [ ]:
plot_lstm_delay_l1_l5_combined(
    actual33_2, lstm_pred33_2,
    target_day=33, dataset_label="dataset2",
    date_label="02_May_2024", output_dir=str(PLOTS_DIR_2)
)


### Ionospheric Delay — Transformer Forecast, Day 33 (G3 Storm)

In [ ]:
plot_transformer_delay_l1_l5_combined(
    actual33_2, trans_pred33_2,
    target_day=33, dataset_label="dataset2",
    date_label="02_May_2024", output_dir=str(PLOTS_DIR_2)
)


### Delay Error Metrics — Day 33 (2 May 2024, G3 Storm)

In [ ]:
iono_actual33_2 = tec_to_iono_delay(actual33_2)
iono_lstm33_2   = tec_to_iono_delay(lstm_pred33_2)
iono_trans33_2  = tec_to_iono_delay(trans_pred33_2)

print("Ionospheric Delay Error (L1, vertical) — Day 33 — Dataset 2 (G3 Storm)")
print("=" * 62)
delay_metrics("LSTM",        iono_lstm33_2,  iono_actual33_2)
delay_metrics("Transformer", iono_trans33_2, iono_actual33_2)


### Save Forecast CSV — Day 33 (2 May 2024)

In [ ]:
save_delay_csv(actual33_2, lstm_pred33_2, trans_pred33_2,
               output_dir=OUTPUT_DIR_2, filename="day33_02May2024_G3_ionospheric_delay.csv")


---
## Full Metrics — Both Storm Days (Day 33 & Day 37)
---

### Storm event threshold (from training set)

In [ ]:
storm_threshold_2 = float(
    np.percentile(daily_matrix_2[1:30], EVENT_THRESHOLD_PERCENTILE)
)
print(f"Dataset 2 threshold ({EVENT_THRESHOLD_PERCENTILE}th percentile): {storm_threshold_2:.2f} TECU")


### RMSE, Correlation, POD, CSI, F1, FAR — Day 37 (G2 Storm)

In [ ]:
lstm_rmse_m37_2        = rmse(lstm_pred37_2, actual37_2)
transformer_rmse_m37_2 = rmse(trans_pred37_2, actual37_2)
lstm_corr_37_2         = pearson_correlation(lstm_pred37_2, actual37_2)
trans_corr_37_2        = pearson_correlation(trans_pred37_2, actual37_2)
lstm_pod_37_2          = probability_of_detection(lstm_pred37_2, actual37_2, threshold=storm_threshold_2)
trans_pod_37_2         = probability_of_detection(trans_pred37_2, actual37_2, threshold=storm_threshold_2)
lstm_csi_37_2          = critical_success_index(lstm_pred37_2, actual37_2, threshold=storm_threshold_2)
trans_csi_37_2         = critical_success_index(trans_pred37_2, actual37_2, threshold=storm_threshold_2)
lstm_f1_37_2           = f1_score(lstm_pred37_2, actual37_2, threshold=storm_threshold_2)
trans_f1_37_2          = f1_score(trans_pred37_2, actual37_2, threshold=storm_threshold_2)
lstm_far_37_2          = false_alarm_ratio(lstm_pred37_2, actual37_2, threshold=storm_threshold_2)
trans_far_37_2         = false_alarm_ratio(trans_pred37_2, actual37_2, threshold=storm_threshold_2)

print("Day 37 — 6 May 2024 — G2 Storm")
print(f"{'Metric':<20} {'LSTM':>12} {'Transformer':>14}")
print("-" * 48)
print(f"{'RMSE (TECU)':<20} {lstm_rmse_m37_2:>12.4f} {transformer_rmse_m37_2:>14.4f}")
print(f"{'Correlation':<20} {lstm_corr_37_2:>12.4f} {trans_corr_37_2:>14.4f}")
print(f"{'POD':<20} {lstm_pod_37_2:>12.4f} {trans_pod_37_2:>14.4f}")
print(f"{'CSI':<20} {lstm_csi_37_2:>12.4f} {trans_csi_37_2:>14.4f}")
print(f"{'F1 Score':<20} {lstm_f1_37_2:>12.4f} {trans_f1_37_2:>14.4f}")
print(f"{'FAR':<20} {lstm_far_37_2:>12.4f} {trans_far_37_2:>14.4f}")


### L1 & L5 Delay Metrics — Day 37 (G2 Storm)

In [ ]:
freqs = {"L1": F_L1, "L5": F_L5}
delay_rows_37 = []

for freq_label, freq_hz in freqs.items():
    actual_delay, lstm_delay, trans_delay = compute_all_delays(
        actual37_2, lstm_pred37_2, trans_pred37_2, frequency=freq_hz
    )
    for model_name, pred_delay in (("LSTM", lstm_delay), ("Transformer", trans_delay)):
        delay_rmse = rmse(pred_delay, actual_delay)
        delay_corr = pearson_correlation(pred_delay, actual_delay)
        delay_rows_37.append({
            "Frequency": freq_label,
            "Model": model_name,
            "RMSE (m)": delay_rmse,
            "Correlation": delay_corr,
        })
        print(f"{freq_label:<3} | {model_name:<12} | RMSE: {delay_rmse:.4f} m | Corr: {delay_corr:.4f}")

iono_delay_metrics_37 = pd.DataFrame(delay_rows_37).round(4)
iono_delay_metrics_37


### RMSE, Correlation, POD, CSI, F1, FAR — Day 33 (G3 Storm)

In [ ]:
lstm_rmse_m33_2        = rmse(lstm_pred33_2, actual33_2)
transformer_rmse_m33_2 = rmse(trans_pred33_2, actual33_2)
lstm_corr_33_2         = pearson_correlation(lstm_pred33_2, actual33_2)
trans_corr_33_2        = pearson_correlation(trans_pred33_2, actual33_2)
lstm_pod_33_2          = probability_of_detection(lstm_pred33_2, actual33_2, threshold=storm_threshold_2)
trans_pod_33_2         = probability_of_detection(trans_pred33_2, actual33_2, threshold=storm_threshold_2)
lstm_csi_33_2          = critical_success_index(lstm_pred33_2, actual33_2, threshold=storm_threshold_2)
trans_csi_33_2         = critical_success_index(trans_pred33_2, actual33_2, threshold=storm_threshold_2)
lstm_f1_33_2           = f1_score(lstm_pred33_2, actual33_2, threshold=storm_threshold_2)
trans_f1_33_2          = f1_score(trans_pred33_2, actual33_2, threshold=storm_threshold_2)
lstm_far_33_2          = false_alarm_ratio(lstm_pred33_2, actual33_2, threshold=storm_threshold_2)
trans_far_33_2         = false_alarm_ratio(trans_pred33_2, actual33_2, threshold=storm_threshold_2)

print("Day 33 — 2 May 2024 — G3 Storm")
print(f"{'Metric':<20} {'LSTM':>12} {'Transformer':>14}")
print("-" * 48)
print(f"{'RMSE (TECU)':<20} {lstm_rmse_m33_2:>12.4f} {transformer_rmse_m33_2:>14.4f}")
print(f"{'Correlation':<20} {lstm_corr_33_2:>12.4f} {trans_corr_33_2:>14.4f}")
print(f"{'POD':<20} {lstm_pod_33_2:>12.4f} {trans_pod_33_2:>14.4f}")
print(f"{'CSI':<20} {lstm_csi_33_2:>12.4f} {trans_csi_33_2:>14.4f}")
print(f"{'F1 Score':<20} {lstm_f1_33_2:>12.4f} {trans_f1_33_2:>14.4f}")
print(f"{'FAR':<20} {lstm_far_33_2:>12.4f} {trans_far_33_2:>14.4f}")


### L1 & L5 Delay Metrics — Day 33 (G3 Storm)

In [ ]:
delay_rows_33 = []

for freq_label, freq_hz in freqs.items():
    actual_delay, lstm_delay, trans_delay = compute_all_delays(
        actual33_2, lstm_pred33_2, trans_pred33_2, frequency=freq_hz
    )
    for model_name, pred_delay in (("LSTM", lstm_delay), ("Transformer", trans_delay)):
        delay_rmse = rmse(pred_delay, actual_delay)
        delay_corr = pearson_correlation(pred_delay, actual_delay)
        delay_rows_33.append({
            "Frequency": freq_label,
            "Model": model_name,
            "RMSE (m)": delay_rmse,
            "Correlation": delay_corr,
        })
        print(f"{freq_label:<3} | {model_name:<12} | RMSE: {delay_rmse:.4f} m | Corr: {delay_corr:.4f}")

iono_delay_metrics_33 = pd.DataFrame(delay_rows_33).round(4)
iono_delay_metrics_33


---
## Summary Metrics Table — Day 33 & Day 37 (Dataset 2, G3 & G2 Storms)
---

In [ ]:
metrics_summary_storms = pd.DataFrame({
    "Dataset":    ["Dataset 2"] * 4,
    "Storm":      ["G3 (2 May)", "G3 (2 May)", "G2 (6 May)", "G2 (6 May)"],
    "Day":        [33, 33, 37, 37],
    "Model":      ["LSTM", "Transformer", "LSTM", "Transformer"],
    "RMSE (TECU)": [lstm_rmse_m33_2, transformer_rmse_m33_2,
                    lstm_rmse_m37_2, transformer_rmse_m37_2],
    "Correlation": [lstm_corr_33_2,  trans_corr_33_2,
                    lstm_corr_37_2,  trans_corr_37_2],
    "POD":        [lstm_pod_33_2,  trans_pod_33_2,
                   lstm_pod_37_2,  trans_pod_37_2],
    "CSI":        [lstm_csi_33_2,  trans_csi_33_2,
                   lstm_csi_37_2,  trans_csi_37_2],
    "F1 Score":   [lstm_f1_33_2,  trans_f1_33_2,
                   lstm_f1_37_2,  trans_f1_37_2],
    "FAR":        [lstm_far_33_2, trans_far_33_2,
                   lstm_far_37_2, trans_far_37_2],
}).round(4)

print("=" * 90)
print("Storm-Day Forecast — Summary Metrics Table (Dataset 2, Days 33 & 37)")
print("=" * 90)
print(metrics_summary_storms.to_string(index=False))

summary_csv_path = GENERATED_PLOTS_DIR / "metrics_summary_day33_day37_dataset2_storms.csv"
GENERATED_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_summary_storms.to_csv(summary_csv_path, index=False)
print(f"\nSaved summary table to: {summary_csv_path}")

metrics_summary_storms
